# 2. Turning Looks Into Numbers
### Stage 2 of 7 — describing eye movements with measurements

---

**How to read this notebook:** This is part of a 7-notebook series that walks through the *entire* research project, step by step, in the same order the actual analysis was done — this one covers stage 2 of 7. Every number and chart here is real, pulled directly from the study's actual data (130 students, 13 puzzles). A few small grey boxes contain code — you don't need to understand the code itself, just run each one (click it, then press Shift+Enter) to see the real result. Nothing needs to be edited.

---

## Overview

A raw eye-tracking recording is a firehose of coordinates changing every fraction of a second — not something you can feed directly into any kind of analysis. This stage turns that firehose into a small set of **meaningful, stable numbers per student per puzzle** — the same way a fitness tracker doesn't report your raw heartbeat waveform, it reports "average heart rate" and "steps taken."

By the end of this step, every one of the 130 students × 13 puzzles = **1,690 attempts** has been boiled down to **35 measurements** each.

## What You'll Learn Here

1. What a "measurement" (researchers call it a *feature*) actually is, and why raw coordinates aren't useful on their own
2. The difference between measuring "time on the right answer" vs. "time on the wrong answers"
3. A handful of specific measurements this study used, and what each one is meant to capture about how someone searches
4. Why pupil size is even part of this at all

## Background, Explained Simply

### What is a "measurement" here?

A measurement is just one number that summarises some aspect of behaviour across a whole puzzle attempt. Instead of thousands of raw eye positions, we ask focused questions like *"what fraction of the time did this student spend looking at the correct spot?"* — one clean number, per student, per puzzle.

### Splitting "correct spot" from "wrong spots"

Every puzzle had exactly one correct hiding place and several decoy spots. For every measurement, this study computed it **twice** — once counting only time/attention spent on the *correct* spot, and again for the *wrong* spots combined. This split matters because it lets you separately ask "how good were they at finding it?" and "how much did wrong options distract them?" — two different things that a single combined number would blur together.

### The specific measurements used

| Measurement | Plain-language meaning |
|---|---|
| **Dwell ratio** (on the correct spot) | Of all the time spent actually looking somewhere on the picture, what share was on the correct spot? The single most important measurement in this whole study. |
| **Time to first look** | How many milliseconds after the puzzle appeared before their eyes first reached the correct spot? |
| **Number of fixations** | How many separate times did their eyes pause on the correct spot? |
| **Scanpath length** | Adding up the total distance their eyes travelled while searching — a short, direct search vs. a long, wandering one. |
| **Re-fixation count** | How many times did they leave the correct spot and then come back to it? (High values can mean double-checking, or losing track and re-finding it.) |
| **Pupil size change** | Whether their pupils got bigger or smaller as the puzzle went on — pupil size is a well-known physical signal of mental effort, so a growing pupil can hint at a puzzle getting harder for that person in real time. |

## Discussion Questions

1. Why measure "dwell ratio" (a fraction) instead of just "total time spent" (a raw number)? What problem does using a fraction solve when comparing students who took very different amounts of total time?
2. If a student found the answer almost instantly, would you expect their re-fixation count to be high or low? What about a student who found it, doubted themselves, and double-checked?
3. Pupil size naturally changes with lighting, not just mental effort. What would you want to control for before trusting pupil size as a measurement of difficulty?

---

## Seeing It for Two Real Students

Here's the idea made concrete: two real students, the same puzzle ("Crown"), turned into two of these measurements. One of them found the hidden crown; the other didn't.

In [ ]:
# Run this cell to compare two real students' measurements on the same puzzle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

feat = pd.read_parquet('../data/processed/features_per_task.parquet')
labels = pd.read_csv('../data/processed/labels.csv')[['participant_id', 'performance_label']]
feat = feat.merge(labels, on='participant_id')

task_df = feat[feat['task'] == 'Crown'].dropna(subset=['correct_aoi_dwell_ratio', 'distractor_dwell_ratio'])
high_row = task_df[task_df['performance_label'] == 1].iloc[0]
low_row = task_df[task_df['performance_label'] == 0].iloc[0]

fig, ax = plt.subplots(figsize=(8, 5))
labels_x = ['Share of time on\nthe correct spot', 'Share of time on\nwrong spots']
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, [high_row['correct_aoi_dwell_ratio'], high_row['distractor_dwell_ratio']],
       width, label=f"Student {high_row['participant_id']} (solved it)", color='#2ecc71')
ax.bar(x + width/2, [low_row['correct_aoi_dwell_ratio'], low_row['distractor_dwell_ratio']],
       width, label=f"Student {low_row['participant_id']} (didn't solve it)", color='#e74c3c')
ax.set_xticks(x); ax.set_xticklabels(labels_x)
ax.set_ylabel('Share of total looking-time')
ax.set_title('Two real students, same puzzle ("Crown") — turned into numbers')
ax.legend()
plt.tight_layout()
plt.show()

Already, just from these two students on one puzzle, a pattern starts to peek through — one that the rest of this series will investigate properly across all 130 students. That's exactly the point of turning "where someone looked" into numbers: it lets you *compare* people and *test* patterns, in a way that staring at 130 separate videos never could.

## What Actually Happened, By the Numbers

Combining the measurements computed straight from the Metrics file (like total time and number of fixations) with the ones computed from the raw gaze stream (like dwell ratio and scanpath length), every attempt ended up with **35 measurements**. Across all 130 students × 13 puzzles, that's a table of **1,690 rows × 37 columns** (35 measurements plus the student's name and which puzzle it was) — small enough to work with easily, but rich enough to capture real behavioural differences.

---

**Next: [03_deciding_high_or_low.ipynb](03_deciding_high_or_low.ipynb)** — now that every attempt has measurements, how do we decide which students count as "strong" problem-solvers and which count as "struggling"?